In [35]:
import pandas as pd
from database_helper import get_engine  # Use the helper we created

# Load the Kaggle CSV into memory
df_kaggle = pd.read_csv('kaggle_income.csv', encoding='ISO-8859-1')

# Clean the column names: 'State Name' becomes 'state_name'
df_kaggle.columns = [c.lower().replace(' ', '_') for c in df_kaggle.columns]

# Push to Neon (this creates the 'salary_data' table)
engine = get_engine()
df_kaggle.to_sql('salary_data', engine, if_exists='replace', index=False)

526

In [36]:
# Query to get the average mean salary per state
sql_query = """
SELECT 
    state_name, 
    AVG("mean") as avg_state_salary
FROM salary_data
GROUP BY state_name
ORDER BY avg_state_salary DESC;
"""

In [37]:
# Force a rollback to clear the "invalid transaction" state
engine.dispose()

In [38]:
# Execute the query and load directly into a new DataFrame
df_analysis = pd.read_sql(sql_query, engine)

In [39]:
# Calculate national average in Pandas
national_avg = df_analysis['avg_state_salary'].mean()

# Transformation: Create a 'Percent Difference' column
df_analysis['diff_from_national_%'] = (
    (df_analysis['avg_state_salary'] - national_avg) / national_avg
) * 100

# Peek at your top 5 highest paying states
print(df_analysis.head())

             state_name  avg_state_salary  diff_from_national_%
0  District of Columbia      90668.421875             38.960469
1           Connecticut      89227.219718             36.751650
2            New Jersey      88657.644144             35.878705
3              Maryland      87689.604096             34.395065
4         Massachusetts      84878.683582             30.086985
